# 01 — Instagram Data Exploration

Exploratory analysis of synthetic Instagram profile data to understand feature distributions before model training.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from services.ml.training.generate_synthetic_data import generate_engagement_data, generate_bot_data

X_eng, y_eng = generate_engagement_data(n=5000)
X_bot, y_bot = generate_bot_data(n=5000)
print('Engagement features shape:', X_eng.shape)
print('Bot features shape:', X_bot.shape)

In [ ]:
df_eng = pd.DataFrame(X_eng, columns=['log_followers', 'log_following', 'log_posts', 'ff_ratio', 'log_avg_likes'])
df_eng['engagement_rate'] = y_eng
df_eng.describe()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), df_eng.columns):
    ax.hist(df_eng[col], bins=50, edgecolor='white', color='#405DE6')
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('')
plt.suptitle('Feature Distributions (Synthetic Data)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
corr = df_eng.corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
df_bot = pd.DataFrame(X_bot, columns=['log_fc', 'log_fwc', 'log_pc', 'ff_ratio', 'er_deviation', 'engagement_rate'])
df_bot['is_bot'] = y_bot.astype(int)
print('Class balance:')
print(df_bot['is_bot'].value_counts())
print()
df_bot.groupby('is_bot')[['log_fc', 'er_deviation', 'engagement_rate']].mean()

## Key findings

- **Engagement rate** decreases with follower count (power-law) — this is the central signal for both the engagement predictor and the bot detector.
- **Bot accounts** show anomalously high `er_deviation` — much lower engagement than expected for their follower count.
- **Feature correlation** between `log_followers` and `log_avg_likes` is strong (~0.7), confirming that bigger accounts generate more absolute engagement but lower relative engagement.
- The follower-following ratio is a weak bot signal alone but becomes powerful in combination with engagement rate deviation.